In [43]:
# ========================================
# 1️⃣ Imports
# ========================================
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings import init_embeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec


# ========================================
# 2️⃣ Carregar variáveis de ambiente
# ========================================
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# ========================================
# 3️⃣ Carregar PDFs
# ========================================
pdf_dir = "pdfs"

loader = PyPDFDirectoryLoader(pdf_dir)
documents = loader.load()

print(f"✅ {len(documents)} páginas carregadas")


# ========================================
# 4️⃣ Split em chunks
# ========================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"✅ {len(chunks)} chunks criados")


# ========================================
# 5️⃣ Criar Embeddings (LangChain moderno)
# ========================================
embeddings = init_embeddings(
    model="text-embedding-3-small",
    provider="openai",
    api_key=OPENAI_API_KEY
)

texts = [chunk.page_content for chunk in chunks]

vectors = embeddings.embed_documents(texts)

print(f"✅ {len(vectors)} embeddings gerados")


# ========================================
# 6️⃣ Conectar ao Pinecone (SDK v7+)
# ========================================
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "bula-index"

# cria índice se não existir
if index_name not in pc.list_indexes().names():
    print(f"Criando índice {index_name}...")

    pc.create_index(
        name=index_name,
        dimension=1536,  # text-embedding-3-small
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

print(f"✅ Conectado ao índice '{index_name}'")


# ========================================
# 7️⃣ Preparar dados para UPSERT
# ========================================


ids = [f"{doc.metadata['source']}-{i}" for i, doc in enumerate(chunks)]

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name,
    ids=ids
)


print("🚀 Vetores enviados com sucesso para o Pinecone!")

✅ 148 páginas carregadas
✅ 727 chunks criados
✅ 727 embeddings gerados
✅ Conectado ao índice 'bula-index'
🚀 Vetores enviados com sucesso para o Pinecone!


In [55]:
import os
from dotenv import load_dotenv

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain.embeddings import init_embeddings
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

load_dotenv()

# ========= PINECONE =========

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "bula-index"
index = pc.Index(index_name)

# ========= EMBEDDINGS =========

embeddings = init_embeddings(
    model="text-embedding-3-small",
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY")
)

# ========= VECTOR STORE =========

vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings
)



llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""Você é um assistente especializado na leitura de bulas de medicamentos.

Sua função é responder dúvidas utilizando SOMENTE as informações
presentes no contexto fornecido.

Regras obrigatórias:
- Seja cordial e claro.
- Utilize exclusivamente o conteúdo do contexto.
- NÃO invente informações.
- NÃO forneça aconselhamento médico.

Contexto:
{context}

Pergunta:
{input}

Resposta:
""")

document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

print("✅ Document chain criada")

# ========= RETRIEVER =========

base_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 12,
        "fetch_k": 30,
        "score_threshold": 0.5
    },
)

retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

print("✅ Retriever pronto")

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

print("✅ Retrieval Chain criada")


✅ Document chain criada
✅ Retriever pronto
✅ Retrieval Chain criada


In [56]:
response = retrieval_chain.invoke({
    "input": "posso consumir alcool e utilizar alenia?"
})

print(response["answer"])

O contexto não fornece informações específicas sobre a interação entre álcool e Alenia. Recomendo que você converse com seu médico ou farmacêutico para esclarecer essa dúvida.
